# CTD Assignment 6

Lesson 6: Data Wrangling and Manipulation

This notebook uses Pandas selection, aggregation, pivot tables, merging, joining, transformation, filtering, sorting, and `apply()`.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


def list_kaggle_files():
    for dirname, _, filenames in os.walk("/kaggle/input"):
        for filename in filenames:
            print(os.path.join(dirname, filename))


def find_csv_with_columns(required_columns, preferred_keywords=None):
    """Return the first Kaggle CSV containing all required columns."""
    required_columns = set(required_columns)
    preferred_keywords = [word.lower() for word in (preferred_keywords or [])]
    matches = []

    for path in Path("/kaggle/input").rglob("*.csv"):
        try:
            preview = pd.read_csv(path, nrows=5)
        except Exception:
            continue

        if required_columns.issubset(preview.columns):
            path_text = str(path).lower()
            score = sum(keyword in path_text for keyword in preferred_keywords)
            matches.append((score, path))

    if not matches:
        raise FileNotFoundError(f"No CSV file found with columns: {sorted(required_columns)}")

    matches.sort(reverse=True)
    return matches[0][1]


list_kaggle_files()


## Task 1
Data Selection

In [ ]:
data1 = {
    "Name": ["Alice", "Bob", "Charlie", "David", "Eva"],
    "Age": [25, 30, 35, 40, 30],
    "Salary": [50000, 60000, 70000, 80000, 55000],
}

data2 = {
    "Name": ["Frank", "Grace", "Helen", "Ian", "Jack"],
    "Age": [28, 33, 35, 29, 40],
    "Salary": [52000, 58000, 72000, 61000, 85000],
}

data3 = {
    "Name": ["Frank", "Helen", "Ian", "Hima", "Chaka"],
    "Age": [17, 93, 12, 57, 106],
    "Favorite Color": ["blue", "pink", "burgundy", "red", "turquoise"],
}

df1 = pd.DataFrame(data1)
df2 = pd.DataFrame(data2)
df3 = pd.DataFrame(data3)

print("df1")
display(df1)

print("df2")
display(df2)

print("df3")
display(df3)

print("Name column from df1")
print(df1["Name"])

print("Name and Salary columns from df1")
display(df1[["Name", "Salary"]])

print("First three rows using iloc")
display(df1.iloc[:3])


## Task 2
Data Aggregation

In [ ]:
salary_by_age = df1.groupby("Age").agg({"Salary": ["mean", "sum", "count"]})

print("Salary aggregation by age")
display(salary_by_age)


## Task 3
Practice Pivot Tables

In [ ]:
ecommerce_path = find_csv_with_columns(
    [
        "Customer_ID",
        "Purchase_Category",
        "Purchase_Amount",
        "Gender",
        "Income_Level",
        "Education_Level",
        "Marital_Status",
    ],
    preferred_keywords=["ecommerce", "consumer", "behavior"],
)

print(f"Reading ecommerce data from: {ecommerce_path}")
ecommerce = pd.read_csv(ecommerce_path)

print("First 5 rows of ecommerce")
display(ecommerce.head())

ecommerce["Purchase_Amount"] = (
    ecommerce["Purchase_Amount"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

buying_patterns = pd.pivot_table(
    ecommerce,
    index="Purchase_Category",
    columns=["Gender", "Income_Level"],
    values="Purchase_Amount",
    aggfunc="sum",
    fill_value=0,
)

print("Buying patterns pivot table")
display(buying_patterns)

demographics = pd.pivot_table(
    ecommerce,
    index=["Income_Level", "Education_Level"],
    columns="Marital_Status",
    values="Customer_ID",
    aggfunc="count",
    fill_value=0,
)

print("Demographics pivot table")
display(demographics)


## Task 4
Practice `apply()`

In [ ]:
jobs_path = find_csv_with_columns(
    ["experience_level", "salary_range", "required_skills"],
    preferred_keywords=["job", "recommendation", "ai"],
)

print(f"Reading jobs data from: {jobs_path}")
jobs = pd.read_csv(jobs_path)

print("First 10 rows of jobs")
display(jobs.head(10))


def salary_number(value):
    cleaned = str(value).replace("$", "").replace(",", "").strip().lower()
    parts = cleaned.replace("to", "-").split("-")
    numbers = []

    for part in parts:
        digits = "".join(character for character in part if character.isdigit() or character == ".")
        if digits:
            number = float(digits)
            if "k" in part:
                number = number * 1000
            numbers.append(number)

    return max(numbers) if numbers else np.nan


def check_job(row):
    experience = str(row["experience_level"]).lower()
    skills = str(row["required_skills"]).lower()
    salary = salary_number(row["salary_range"])

    is_entry_level = "entry" in experience
    high_salary = salary >= 70000
    has_sql_and_python = "sql" in skills and "python" in skills

    if is_entry_level and high_salary and has_sql_and_python:
        return "Yes"
    return "No"


jobs["Check These Out"] = jobs.apply(check_job, axis=1)
my_jobs = jobs[jobs["Check These Out"] == "Yes"]

print("Entry-level jobs paying at least 70000 and requiring SQL plus Python")
display(my_jobs.head(10))


## Task 5
Merging and Joining DataFrames

In [ ]:
df_1_3_merged = pd.merge(
    df1,
    df3,
    on="Name",
    how="outer",
    suffixes=["_left", "_right"],
)

print("Outer merge of df1 and df3")
display(df_1_3_merged)

df_1_3_merged["Salary"] = df_1_3_merged["Salary"].fillna(15000)
df_1_3_merged["Favorite Color"] = df_1_3_merged["Favorite Color"].fillna("yellow")

print("After filling missing Salary and Favorite Color values")
display(df_1_3_merged)

df_1_3_merged["Age"] = np.where(
    df_1_3_merged["Age_left"].notna(),
    df_1_3_merged["Age_left"],
    df_1_3_merged["Age_right"],
)

print("After creating one combined Age column")
display(df_1_3_merged)

df_1_3_merged = df_1_3_merged.drop(["Age_left", "Age_right"], axis=1)

print("After dropping Age_left and Age_right")
display(df_1_3_merged)

df1_b = df1.set_index("Name")
df3_b = df3.set_index("Name")

joined_df = df1_b.join(df3_b, how="outer", lsuffix="_left", rsuffix="_right")

print("Outer join of df1_b and df3_b")
display(joined_df)


## Task 6
Filtering Rows Based on Conditions

In [ ]:
older_than_30 = df1[df1["Age"] > 30]

print("Rows where Age is greater than 30")
display(older_than_30)


## Task 7
Sorting Data

In [ ]:
df1_sorted_by_salary = df1.sort_values(by="Salary", ascending=False)

print("df1 sorted by Salary from highest to lowest")
display(df1_sorted_by_salary)


## Task 8
Renaming Columns

In [ ]:
df1_renamed = df1.rename(columns={"Age": "Employee Age", "Salary": "Employee Salary"})

print("df1 with renamed columns")
display(df1_renamed)


## Task 9
Data Transformation

In [ ]:
df1_salary_increase = df1.copy()
df1_salary_increase["Salary"] = df1_salary_increase["Salary"] * 1.10

print("df1 after increasing Salary by 10%")
display(df1_salary_increase)


## Task 10
Concatenating DataFrames

In [ ]:
df1_df2_combined = pd.concat([df1, df2], ignore_index=True)

print("df1 and df2 concatenated")
display(df1_df2_combined)


## Task 11
Data Wrangling a Kaggle Dataset

In [ ]:
football_path = find_csv_with_columns(
    ["date", "home_team", "away_team", "home_score", "away_score"],
    preferred_keywords=["international", "football", "results"],
)

print(f"Reading football results from: {football_path}")
football_results = pd.read_csv(football_path)

print("First 5 rows of original football results")
display(football_results.head())

results_1 = football_results[["home_team", "away_team", "home_score", "away_score", "date"]]

print("results_1")
display(results_1.head())

results_2 = results_1.rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "points_for",
        "away_score": "points_against",
    }
)

print("Home team results")
display(results_2.head())

results_3 = results_1.rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "points_for",
        "home_score": "points_against",
    }
)

print("Away team results")
display(results_3.head())

football_results = pd.concat([results_2, results_3], ignore_index=True)

print("Combined football results by team")
display(football_results.head())

points_against = football_results.groupby("team")["points_against"].mean()
points_against = points_against.sort_values(ascending=False)

print("Teams with the highest average points against")
display(points_against.head(10))


## Task 12
More Data Wrangling for Football Results

In [ ]:
football_results["date"] = pd.to_datetime(football_results["date"])

tunisia_games = football_results[football_results["team"] == "Tunisia"].sort_values(
    by="date",
    ascending=False,
)

print("Most recent 10 games for Tunisia")
display(tunisia_games.head(10))


## Task 13
Final Project Week 6 Progress

Paste this section into your separate final project notebook and update the `project_df` loading cell to match your chosen final-project dataset. The code below is written as a strong template: it performs filtering, selection, string transformations, missing-data handling, aggregations, and two engineered features.


In [ ]:
# Replace this file search with the dataset you chose for your final project.
# This example reuses the ecommerce dataset only as a template for the required Week 6 work.
project_df = ecommerce.copy()

print("Project dataset preview")
display(project_df.head())

project_df = project_df.drop_duplicates().copy()

project_df["Purchase_Category"] = project_df["Purchase_Category"].str.strip().str.title()
project_df["Income_Level"] = project_df["Income_Level"].str.strip().str.title()
project_df["Education_Level"] = project_df["Education_Level"].str.strip()

project_df["Purchase_Amount"] = project_df["Purchase_Amount"].fillna(project_df["Purchase_Amount"].median())
project_df["Frequency_of_Purchase"] = project_df["Frequency_of_Purchase"].fillna(
    project_df["Frequency_of_Purchase"].median()
)

selected_columns = [
    "Customer_ID",
    "Age",
    "Gender",
    "Income_Level",
    "Education_Level",
    "Purchase_Category",
    "Purchase_Amount",
    "Frequency_of_Purchase",
    "Customer_Satisfaction",
]
project_subset = project_df.loc[project_df["Purchase_Amount"] >= 100, selected_columns]

print("Filtered and selected project data")
display(project_subset.head())

category_summary = project_df.groupby("Purchase_Category").agg(
    total_purchase_amount=("Purchase_Amount", "sum"),
    average_purchase_amount=("Purchase_Amount", "mean"),
    customer_count=("Customer_ID", "count"),
)

print("Aggregation 1: purchase summary by category")
display(category_summary.sort_values("total_purchase_amount", ascending=False).head(10))

income_summary = project_df.groupby("Income_Level").agg(
    average_satisfaction=("Customer_Satisfaction", "mean"),
    average_purchase_amount=("Purchase_Amount", "mean"),
    purchase_count=("Customer_ID", "count"),
)

print("Aggregation 2: purchase and satisfaction summary by income level")
display(income_summary)

project_df["Average_Amount_Per_Purchase"] = (
    project_df["Purchase_Amount"] / project_df["Frequency_of_Purchase"].replace(0, np.nan)
)

project_df["Spending_Level"] = pd.cut(
    project_df["Purchase_Amount"],
    bins=3,
    labels=["Low", "Medium", "High"],
)

project_df["High_Value_Customer"] = project_df["Purchase_Amount"].map(
    lambda amount: "Yes" if amount >= project_df["Purchase_Amount"].quantile(0.75) else "No"
)

print("Project data with engineered features")
display(project_df[[
    "Customer_ID",
    "Purchase_Amount",
    "Frequency_of_Purchase",
    "Average_Amount_Per_Purchase",
    "Spending_Level",
    "High_Value_Customer",
]].head(10))


### Task 13 Summary

For my Week 6 final project progress, I cleaned the dataset by removing duplicate records and filling missing numeric values with median values. I selected the columns most relevant to customer purchasing behavior, filtered the data to focus on purchases of at least 100 dollars, and standardized string columns such as purchase category and income level.

I used `groupby()` to summarize total and average purchase amount by purchase category, and I also summarized average satisfaction and average purchase amount by income level. These aggregations help show which product categories produce the most revenue and whether spending patterns differ across income groups.

I engineered new features to support analysis: `Average_Amount_Per_Purchase`, `Spending_Level`, and `High_Value_Customer`. These transformations make it easier to compare customers by purchase intensity and identify higher-value customer segments.
